# 6D Pose Estimation - Results Viewer

**Team:** Ulugbek Rakhmatullaev, Karim Shalaby, Mohammad Fakih

This notebook provides an interactive way to view and compare results from our 6D pose estimation models.

## Models:
- **RGB Model** - 4-channel input (RGB + Mask)
- **RGBD Model** - 5-channel input (RGB + Depth + Mask) with z_sensor offset

In [ ]:
import os
import sys

# Setup paths
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

---
## 1. Training History & Curves

View training progress for both models.

In [ ]:
import json
import matplotlib.pyplot as plt

# Load training histories
histories = {}
for model_name in ['rgb', 'rgbd']:
    history_path = f'weights_{model_name}/training_history.json'
    if os.path.exists(history_path):
        with open(history_path, 'r') as f:
            histories[model_name.upper()] = json.load(f)
        print(f"Loaded {model_name.upper()} history: {len(histories[model_name.upper()]['train_loss'])} epochs")
    else:
        print(f"No history found for {model_name.upper()}")

# Plot comparison
if histories:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = {'RGB': 'blue', 'RGBD': 'green'}
    
    for name, hist in histories.items():
        epochs = range(1, len(hist['train_loss']) + 1)
        color = colors.get(name, 'gray')
        
        axes[0, 0].plot(epochs, hist['train_loss'], label=f'{name} Train', color=color, linestyle='-')
        axes[0, 0].plot(epochs, hist['val_loss'], label=f'{name} Val', color=color, linestyle='--')
        
        axes[0, 1].plot(epochs, hist['val_add'], label=name, color=color)
        
        acc_key = 'val_acc_2cm' if 'val_acc_2cm' in hist else 'val_acc'
        if acc_key in hist:
            axes[1, 0].plot(epochs, hist[acc_key], label=name, color=color)
        
        axes[1, 1].plot(epochs, hist['lr'], label=name, color=color)
    
    axes[0, 0].set_title('Loss'); axes[0, 0].legend(); axes[0, 0].grid(True)
    axes[0, 1].set_title('Validation ADD (mm)'); axes[0, 1].legend(); axes[0, 1].grid(True)
    axes[1, 0].set_title('Validation ACC@2cm (%)'); axes[1, 0].legend(); axes[1, 0].grid(True)
    axes[1, 1].set_title('Learning Rate'); axes[1, 1].legend(); axes[1, 1].grid(True)
    axes[1, 1].set_yscale('log')
    
    plt.tight_layout()
    plt.suptitle('Training Comparison: RGB vs RGBD', y=1.02, fontsize=14, fontweight='bold')
    plt.show()

---
## 2. RGB Model Inference

Run inference on a random test image using the RGB model.

In [ ]:
%run scripts/inference/inference_rgb.py

---
## 3. RGBD Model Inference

Run inference using the RGBD model with depth sensor input.

In [ ]:
%run scripts/inference/inference_rgbd.py

---
## 4. Model Comparison (Test Set Metrics)

Compare ADD error and accuracy across both models on the full test set.

In [ ]:
%run scripts/visualization/compare_all_models.py

---
## 5. YOLO Detection Demo

Visualize YOLO segmentation detection results.

In [ ]:
%run scripts/visualization/visualize_yolo.py

---
## 6. Summary Statistics

Print final training statistics for each model.

In [ ]:
print("="*60)
print("FINAL TRAINING STATISTICS")
print("="*60)

for model_name in ['RGB', 'RGBD']:
    if model_name in histories:
        hist = histories[model_name]
        n_epochs = len(hist['train_loss'])
        best_add = min(hist['val_add'])
        best_epoch = hist['val_add'].index(best_add) + 1
        
        acc_key = 'val_acc_2cm' if 'val_acc_2cm' in hist else 'val_acc'
        best_acc = max(hist[acc_key]) if acc_key in hist else 0
        
        print(f"\n{model_name} Model:")
        print(f"  Epochs trained: {n_epochs}")
        print(f"  Best ADD error: {best_add:.2f} mm (epoch {best_epoch})")
        print(f"  Best ACC@2cm:   {best_acc:.1f}%")
        print(f"  Final loss:     {hist['train_loss'][-1]:.4f}")

print("\n" + "="*60)

---
## Architecture Comparison

| Feature | RGB Model | RGBD Model |
|---------|-----------|------------|
| **Input** | RGB + Mask (4ch) | RGB + Depth + Mask (5ch) |
| **Backbone** | ResNet50 | Dual-stream (ResNet50 + DepthCNN) |
| **Z Prediction** | Absolute depth | z_sensor + z_offset |
| **X,Y** | Geometric | Geometric |
| **Depth Prior** | None | Uses sensor measurement |

### Key Innovation: z_offset Prediction

The RGBD model predicts a small offset from the sensor measurement:
```python
z_sensor = median(depth[mask > 0.5])  # Sample from object pixels
z_final = z_sensor + z_offset         # Model predicts correction
```

This is easier to learn than absolute depth estimation!